In [1]:
!conda install -c conda-forge pdfplumber pdf2image pytesseract poppler pandas -y
!pip install Pillow
!pip install opencv-python
!pip install python-docx
print("Installation complete.")

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.0

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.

Installation complete.


In [1]:
from ocr_preprocess import read_pdf_improved
text = read_pdf_improved("Document/test4.pdf")
print(text)


📄 Loading: Document/test4.pdf
   Found 4 pages

📸 Processing Page 1/4...
   ↳ Upscaling 2x...
   ↳ Auto-detected: scan document (noise=238.7, contrast=57.0, white=92.1%)
   ↳ Pipeline: denoise → CLAHE → adaptive → deskew → cleanup

📸 Processing Page 2/4...
   ↳ Upscaling 2x...
   ↳ Auto-detected: clean document (noise=245.9, contrast=60.9, white=87.5%)
   ↳ Pipeline: adaptive threshold (light touch)

📸 Processing Page 3/4...
   ↳ Upscaling 2x...
   ↳ Auto-detected: scan document (noise=248.7, contrast=55.6, white=92.8%)
   ↳ Pipeline: denoise → CLAHE → adaptive → deskew → cleanup

📸 Processing Page 4/4...
   ↳ Upscaling 2x...
   ↳ Auto-detected: scan document (noise=142.1, contrast=43.0, white=95.8%)
   ↳ Pipeline: denoise → CLAHE → adaptive → deskew → cleanup

✅ Extraction complete!
--- Page 1 ---
© ន  ព្រាះរាជាណាបក្រកម្ពុជា
YN:  ជាតិ សាសនា ព្រះមហាក្សត្រ
ហៀ
ក្រសួងអប់រំ យុវជន និងកីឡា
សន្និបាត (២០២៥)
ឯកសារទស្សនាទាន
ស្តីពី
ការអភិវឌ្ឍមូលធនយុវជនតាមរយៈសកម្មភតាពអាទិតភាព
ៃៃអគ្គនាយកដ្ឋានយុវជន


In [2]:
import os
import pytesseract
from pdf2image import convert_from_path
from docx import Document
import cv2
import numpy as np
from PIL import Image, ImageDraw

In [3]:
# --- CONFIGURATION ---
# Tesseract Path (Auto-detect for Mac)
try:
    conda_prefix = os.environ.get('CONDA_PREFIX')
    pytesseract.pytesseract.tesseract_cmd = f"{conda_prefix}/bin/tesseract"
except:
    pytesseract.pytesseract.tesseract_cmd = r'/opt/homebrew/bin/tesseract'

# --- 1. IMPROVED IMAGE CLEANING FUNCTION ---
def clean_image(pil_image):
    img = np.array(pil_image)

    # 1. Convert to Grayscale
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img

    # 2. Resize (Upscale) - Making text bigger helps Tesseract
    # Scaling by 2x
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # 3. Adaptive Thresholding (Better for varying lighting/text weight)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 2)

    return Image.fromarray(binary)

In [4]:
def read_docx_file(filename):
    """Reads text from a modern Word file (.docx)"""
    print(f"📄 Reading DOCX: {filename}")
    try:
        doc = Document(filename)
        full_text = []
        for para in doc.paragraphs:
            if para.text.strip(): # Skip empty lines
                full_text.append(para.text)
        return "\n".join(full_text)
    except Exception as e:
        return f"Error reading DOCX: {e}"

In [5]:
def read_pdf_file(filename):
    """Reads text from PDF using Tesseract OCR"""
    print(f"📸 Scanning PDF: {filename} (This might take time...)")
    extracted_text = ""

    try:
        # Convert PDF to Images (300 DPI is best for Tesseract)
        images = convert_from_path(filename, dpi=300)

        for i, img in enumerate(images):
            print(f"   - Processing Page {i+1}...")

            # Clean the image (Pre-processing)
            processed_img = clean_image(img)

            # Run Tesseract
            # psm 6 = Assume a single uniform block of text
            # Added 'eng' to lang to support numbers and URLs
            config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(processed_img, lang='khm+eng', config=config)

            extracted_text += f"--- Page {i+1} ---\n{text}\n"

        return extracted_text
    except Exception as e:
        return f"Error reading PDF: {e}"

In [6]:
def extract_text_from_file(filepath):
    # Check extension
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".docx":
        return read_docx_file(filepath)

    elif ext == ".pdf":
        return read_pdf_file(filepath)

    elif ext == ".doc":
        return "❌ Error: Python cannot read old '.doc' files directly.\n👉 Please open it in Word and 'Save As' -> .docx"

    else:
        return f"❌ Unsupported file type: {ext}"

In [7]:
# --- 4. RUN IT ---

# Change this to your file name
my_file = "Document/docx_1.pdf"
# my_file = "homework.docx"

print("🚀 Starting Extraction...")
result = extract_text_from_file(my_file)

print("\n" + "="*40)
print("📝 FINAL EXTRACTED TEXT:")
print("="*40)
print(result)

# Save to file
with open("tesseract_output.txt", "w", encoding="utf-8") as f:
    f.write(result)
print("\n✅ Saved to 'tesseract_output.txt'")

🚀 Starting Extraction...
📸 Scanning PDF: Document/docx_1.pdf (This might take time...)
   - Processing Page 1...
   - Processing Page 2...
   - Processing Page 3...
   - Processing Page 4...
   - Processing Page 5...

📝 FINAL EXTRACTED TEXT:
--- Page 1 ---
ព្រះរាជាណាចក្រកម្ពជា
ជាតិ សាសនា ព្រះមហាក្សត្រ
ក្រសួងអប់រំ យុវជន និងកីឡា
សុន្ទរកថា
ឯកឧត្តមបណ្ឌិតសភាចារ្យ ហង់ជួន ណារ៉ុន ឧបនាយករដ្ឋមន្ត្រី
រដ្ឋមន្ត្រីក្រសួងអប់រំ យុវជន និងកីឡា
ថ្លែងក្នងទិវាគ្រូបង្រៀន ៥ តុលា ២០២៥
ក្រោមប្រធានបទ
គ្រូបង្រៀនជាគួអង្គសំខាន់នៃបរិវត្តកម្មបញ្ញាសិប្បនិម្មិត*
វិទ្យាស្ថានបច្ចេកវិទ្យាកម្ពជា, ថ្ងៃទី៦ ខែតុលា ឆ្នាំ២០២៥

9 ឯកឧត្តម លោកជំទាវថ្នាក់ដឹកនាំក្រសួងអប់រំ យុវជន និងកីឡា

9 ឯកឧត្តម លោកជំទាវលោក្មលោកស្រី ភ្ញៀវកិត្តិយសជាតិ និងអន្តរជាតិ

9 លោកគ្រូអ្នកគ្រូ HANG ជាទិស្រឡាញ់រាបំអាន!
ថ្ងៃនេះ ខ្ញុំមានសេចក្តីរីករាយដោយបានចូលរួម និងអបអរសាទរជាមួយ ឯកឧត្តម លោកជំទាវ លោក
លោកស្រី លោកគ្រូ អ្នកគ្រូទាំងអស់ នៅក្នងទិវាគ្រូបង្រៀន ៥ តុលា ២០២៥ ក្រោមប្រធានបទ
គ្រួបង្រៀនជាតួអង្គសំខាន់នៃបរិវត្តកម្មសំប្បនិម្មិត» ដែលបានរៀបចំយ៉ាងឱឡារិកនាពេលនេះ។
ក្ន

In [8]:
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import cv2
import numpy as np
from PIL import Image, ImageDraw

# --- CONFIGURATION ---
# Tesseract Path
try:
    conda_prefix = os.environ.get('CONDA_PREFIX')
    pytesseract.pytesseract.tesseract_cmd = f"{conda_prefix}/bin/tesseract"
except:
    pytesseract.pytesseract.tesseract_cmd = r'/opt/homebrew/bin/tesseract'

# --- FUNCTION ---
def find_suspicious_words(image, config=''):
    # Get detailed data (words, positions, confidence)
    # Use DICT and convert to DataFrame
    # ADDED 'eng' to lang to help with numbers/URLs
    data_dict = pytesseract.image_to_data(image, lang='khm+eng', config=config, output_type=pytesseract.Output.DICT)
    data = pd.DataFrame(data_dict)

    # Ensure conf is numeric
    data['conf'] = pd.to_numeric(data['conf'], errors='coerce')

    # Filter: Keep only actual text (remove empty space)
    text_data = data[data.conf != -1]

    # Filter: Keep only "Low Confidence" words (likely wrong)
    # We set the threshold at 60%. Anything below 60 is probably garbage.
    suspicious = text_data[text_data['conf'] < 60]

    # Return text, conf, AND coordinates for drawing
    return suspicious[['text', 'conf', 'left', 'top', 'width', 'height']]

# --- RUN IT ---
pdf_filename = "Document/docx_1.pdf"

print(f"🚀 Analyzing PDF for errors: {pdf_filename}")

# Convert PDF to Image (Higher DPI = Better Quality)
images = convert_from_path(pdf_filename, dpi=300)

# Analyze Page 1
print("   Scanning Page 1...")

# 1. Preprocess (Resize + Clean)
# We can now use the global clean_image function since it's updated
clean_img = clean_image(images[0])

# 2. Find errors
# Added config to match the main extraction logic
custom_config = r'--oem 3 --psm 6'
bad_words_table = find_suspicious_words(clean_img, config=custom_config)

# Show the result
print("\n" + "="*40)
print("🚩 SUSPICIOUS WORDS (Likely Wrong):")
print("="*40)

if not bad_words_table.empty:
    # Print the top 20 worst words
    print(bad_words_table[['text', 'conf']].head(20).to_string(index=False))

    # Save to CSV for your report
    bad_words_table.to_csv("ocr_error_report.csv", index=False)
    print("\n💾 Saved full report to 'ocr_error_report.csv'")

    # 3. Draw boxes on the image
    print("   Drawing red boxes around suspicious words...")

    # Convert to RGB so we can draw colored boxes
    visual_img = clean_img.convert("RGB")
    draw = ImageDraw.Draw(visual_img)

    for _, row in bad_words_table.iterrows():
        x, y, w, h = row['left'], row['top'], row['width'], row['height']
        draw.rectangle([x, y, x + w, y + h], outline="red", width=3)

    visual_img.save("suspicious_words_highlighted.jpg")
    print("🖼️ Saved visualized errors to 'suspicious_words_highlighted.jpg'")

else:
    print("✅ Amazing! No low-confidence words found.")

🚀 Analyzing PDF for errors: Document/docx_1.pdf
   Scanning Page 1...

🚩 SUSPICIOUS WORDS (Likely Wrong):
                                                   text  conf
គ្រូបង្រៀនជាគួអង្គសំខាន់នៃបរិវត្តកម្មបញ្ញាសិប្បនិម្មិត*    50
                                                      9    50
                                   លោកជំទាវលោក្មលោកស្រី     0
                                    ជាទិស្រឡាញ់រាបំអាន!     0
                      ខ្ញុំមានសេចក្តីរីករាយដោយបានចូលរួម    34
     គ្រួបង្រៀនជាតួអង្គសំខាន់នៃបរិវត្តកម្មសំប្បនិម្មិត»    29
                    ក្នងការបិទបញ្ចប់ដំណេរការឆ្នាំស័ក្សា    34
                                         បេក្ខជននិទ្ទេស    56
                  តាមរយៈស្តង់ដាសាលារៀនគំរូនៅគ្រប់កម្រិត    55
                  ដែលជាការរំពុកគុណគ្រូបង្រៀនគ្រប់ជំនាន់    36
                          តស៊ូលះបង់ក្នងការបង្ហាត់បង្រៀន    25
                         និងគ្រប់មធ្យោបាយដល់ក្លូយៗសិស្ស    26
                                        ការរំពូកគុណគ្រូ    21
        និងការដឹងគុណគ្រូបា

In [9]:
#Docx to text

import os
from docx import Document

def read_docx_advanced(file_path):
    print(f"📄 Reading DOCX file: {file_path}")

    try:
        # Load the document
        doc = Document(file_path)

        full_text = []

        # 1. READ PARAGRAPHS (Standard text)
        for para in doc.paragraphs:
            if para.text.strip(): # Skip empty lines
                full_text.append(para.text)

        # 2. READ TABLES (Important for assignments!)
        # Many students put info inside tables.
        if len(doc.tables) > 0:
            print(f"   (Found {len(doc.tables)} tables - extracting them...)")
            for table in doc.tables:
                for row in table.rows:
                    row_text = [cell.text.strip() for cell in row.cells if cell.text.strip()]
                    if row_text:
                        # Join cell text with a pipe | to keep structure
                        full_text.append(" | ".join(row_text))

        return "\n".join(full_text)

    except Exception as e:
        return f"❌ Error reading .docx: {e}"

# --- TEST IT ---
# Create a dummy .docx file name or use a real one
filename = "Document/docx_1.docx"

# Check if file exists before running (to avoid error)
if os.path.exists(filename):
    text = read_docx_advanced(filename)
    print("\n" + "="*40)
    print("📝 DOCX CONTENT:")
    print("="*40)
    print(text)
else:
    print(f"⚠️ Please create or upload a file named '{filename}' to test this.")

📄 Reading DOCX file: Document/docx_1.docx

📝 DOCX CONTENT:
ព្រះរាជាណាចក្រកម្ពុជា
ជាតិ សាសនា ព្រះមហាក្សត្រ
ក្រសួងអប់រំ យុវជន និងកីឡា
សុន្ទរកថា
ឯកឧត្តមបណ្ឌិតសភាចារ្យ ហង់ជួន ណារ៉ុន ឧបនាយករដ្ឋមន្ត្រី
រដ្ឋមន្ត្រីក្រសួងអប់រំ យុវជន និងកីឡា
ថ្លែងក្នុងទិវាគ្រូបង្រៀន ៥ តុលា ២០២៥
ក្រោមប្រធានបទ
“គ្រូបង្រៀនជាតួអង្គសំខាន់នៃបរិវត្តកម្មបញ្ញាសិប្បនិម្មិត”
វិទ្យាស្ថានបច្ចេកវិទ្យាកម្ពុជា, ថ្ងៃទី៦ ខែតុលា ឆ្នាំ២០២៥
ឯកឧត្តម លោកជំទាវថ្នាក់ដឹកនាំក្រសួងអប់រំ យុវជន និងកីឡា
ឯកឧត្តម លោកជំទាវ លោក លោកស្រី ភ្ញៀវកិត្តិយសជាតិ និងអន្តរជាតិ
លោកគ្រូ អ្នកគ្រូ អង្គពិធី ជាទីស្រឡាញ់រាប់អាន!
ថ្ងៃនេះ ខ្ញុំមានសេចក្តីរីករាយដោយបានចូលរួម និងអបអរសាទរជាមួយ ឯកឧត្តម លោកជំទាវ លោក លោកស្រី លោកគ្រូ អ្នកគ្រូទាំងអស់ នៅក្នុងទិវាគ្រូបង្រៀន ៥ តុលា ២០២៥ ក្រោមប្រធានបទ “គ្រូបង្រៀនជាតួអង្គសំខាន់នៃបរិវត្តកម្មសិប្បនិម្មិត” ដែលបានរៀបចំយ៉ាងឱឡារិកនាពេលនេះ។
ក្នុងនាមថ្នាក់ដឹកនាំក្រសួងអប់រំ យុវជន និងកីឡា ខ្ញុំសូមកោតសរសើរ និងវាយតម្លៃខ្ពស់ ចំពោះថ្នាក់ដឹកនាំ និងបុគ្គលិកនៃក្រសួងអប់រំ យុវជន និងកីឡា ដែលបានអនុវត្តបេសកកម្មរបស់ខ្លួនប្រកបដោយជោគជ័យគួរឱ្យកត់សម្គាល់ ក

In [10]:
import cv2
import pytesseract
import re
import os
from PIL import Image

# --- CONFIGURATION ---
try:
    conda_prefix = os.environ.get('CONDA_PREFIX')
    pytesseract.pytesseract.tesseract_cmd = f"{conda_prefix}/bin/tesseract"
except:
    pytesseract.pytesseract.tesseract_cmd = r'/opt/homebrew/bin/tesseract'

# ==========================================
# 1. IMAGE CLEANING (Adaptive Thresholding)
# ==========================================
def clean_image_adaptive(image_path):
    img = cv2.imread(image_path)
    if img is None: return None

    # Resize to standard width
    h, w = img.shape[:2]
    scale = 1000 / w
    img = cv2.resize(img, None, fx=scale, fy=scale)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Adaptive Thresholding is BETTER for ID Cards with holograms/shadows
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 2)

    return Image.fromarray(binary)

# ==========================================
# 2. PASS 1: ENGLISH (Robust MRZ + Visual)
# ==========================================
def get_english_data_robust(img):
    print("   ...Pass 1: English & MRZ...")
    # Allow letters, numbers, and common noise chars like < ( )
    custom_config = r'--oem 3 --psm 6'
    text = pytesseract.image_to_string(img, lang='eng', config=custom_config)

    data = {}

    # --- A. ID NUMBER ---
    # Try MRZ first (IDKHM...)
    mrz_id = re.search(r'IDKHM(\d{9})', text)
    if mrz_id:
        data['ID_Number'] = mrz_id.group(1)
    else:
        # Visual fallback
        vis_id = re.search(r'\b\d{9}\b', text)
        data['ID_Number'] = vis_id.group(0) if vis_id else "Not Found"

    # --- B. NAME (EN) ---
    # Strategy 1: MRZ "TIT<<SAMOL" (Allow for OCR errors like K or spaces)
    # This regex means: "Letters, then 1 or more separator (< or K or space), then Letters"
    mrz_name = re.search(r'([A-Z]+)[<K\s]{2,}([A-Z]+)', text)

    if mrz_name:
        data['Name_EN'] = f"{mrz_name.group(1)} {mrz_name.group(2)}"
    else:
        # Strategy 2: Visual Scan (Look for the line with ALL CAPS)
        lines = text.split('\n')
        for line in lines:
            clean = line.strip()
            # If line is all UPPERCASE, has spaces, and is NOT the MRZ code
            if re.match(r'^[A-Z\s]+$', clean) and len(clean) > 5:
                if "KINGDOM" not in clean and "CAMBODIA" not in clean and "IDKHM" not in clean:
                    data['Name_EN'] = clean
                    break

    # --- C. DATE OF BIRTH ---
    # Look for the MRZ date pattern: 6 digits followed by a letter or number
    mrz_dob = re.search(r'\b(\d{2})(\d{2})(\d{2})\d', text)
    if mrz_dob:
        yy, mm, dd = mrz_dob.groups()
        # Filter: Ensure month is 01-12 and day is 01-31
        if 1 <= int(mm) <= 12 and 1 <= int(dd) <= 31:
             prefix = "19" if int(yy) > 40 else "20"
             data['Date_of_Birth'] = f"{dd}.{mm}.{prefix}{yy}"

    return data

# ==========================================
# 3. PASS 2: KHMER (Contextual Search)
# ==========================================
def get_khmer_data_robust(img):
    print("   ...Pass 2: Khmer Context...")
    text = pytesseract.image_to_string(img, lang='khm', config='--oem 3 --psm 6')
    data = {}
    lines = [line.strip() for line in text.split('\n') if line.strip()]

    # Strategy: Find lines that contain specific keywords
    for i, line in enumerate(lines):
        # Place of Birth usually follows "កំណើត" or "ទីកន្លែង"
        if "កំណើត" in line or "ទីកន្លែង" in line:
            # Clean up the label to leave just the value
            # Regex removes the label chars
            val = re.sub(r'[^\u1780-\u17FF\s]', '', line) # Keep only Khmer
            # Heuristic: The value is usually the LAST part of the line
            if len(val) > 5:
                 data['Place_of_Birth'] = val
            elif i + 1 < len(lines):
                 # If value is on next line
                 data['Place_of_Birth'] = lines[i+1]

    return data

# ==========================================
# 4. MAIN EXECUTION
# ==========================================
filename = "Document/camb-id-image-6.png"

if os.path.exists(filename):
    print(f"🚀 Processing: {filename}")

    # Use Adaptive Cleaning this time
    clean_img = clean_image_adaptive(filename)

    eng_data = get_english_data_robust(clean_img)
    khm_data = get_khmer_data_robust(clean_img)

    final_data = {**eng_data, **khm_data}

    print("\n" + "="*40)
    print("🏆 ROBUST RESULT:")
    print("="*40)
    print(f"🆔 ID Number:     {final_data.get('ID_Number', 'Not Found')}")
    print(f"👤 Name (EN):     {final_data.get('Name_EN', 'Not Found')}")
    print(f"🎂 Date of Birth: {final_data.get('Date_of_Birth', 'Not Found')}")
    print(f"📍 Place of Birth:{final_data.get('Place_of_Birth', 'Not Found')}")
else:
    print("⚠️ File not found.")

🚀 Processing: Document/camb-id-image-6.png
   ...Pass 1: English & MRZ...
   ...Pass 2: Khmer Context...

🏆 ROBUST RESULT:
🆔 ID Number:     Not Found
👤 Name (EN):     KHNK K
🎂 Date of Birth: Not Found
📍 Place of Birth:Not Found
